# Track 2 — Intelligent Well Archive: evidence store and router (batches 1-2)

Runs top to bottom with no API key and no network. Steps: config check → build (or load) the evidence store →
locator statistics → route all 32 questions → routing report.

Answering is a later step; nothing here calls an LLM.

In [ ]:
import sys
from pathlib import Path

# Make `archive_qa` importable whether Jupyter starts in submission/ or in the repo root.
here = Path.cwd()
pkg_parent = here if (here / "archive_qa").is_dir() else here / "submission"
sys.path.insert(0, str(pkg_parent))

import pandas as pd
from archive_qa import config, route, store
from archive_qa.locate import locate_stats

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

## 1. Config check

In [ ]:
status = config.check()
print(status)
assert status["DATA_ROOT"], status.get("error")  # lists every path tried
assert status["xml_reports"] == 32 and status["pdf_reports"] == 32 and status["questions_csv"]

## 2. Build the evidence store (or load it if already built)
Set `REBUILD = True` to force a rebuild; the result is identical either way.

In [ ]:
REBUILD = False

if REBUILD or not config.RECORDS_PATH.exists():
    records = store.build_store(verbose=True)
else:
    records = store.load_store()
    print(f"Loaded {len(records)} records from {config.RECORDS_PATH.name}")
records.groupby("section").size().rename("records").to_frame()

## 3. PDF locator statistics

In [ ]:
stats = locate_stats(records)

In [ ]:
# Evidence trail for one record: XML value + where it is printed in the PDF.
k = records[(records.well == "15/9-19-A") & (records.report_date == "1980-01-01") & (records.field == "mdKickoff")]
k[["record_id", "value", "unit", "source_file", "pdf_file", "pdf_page", "pdf_section", "pdf_match", "text"]].T

## 4. Route all 32 questions

In [ ]:
questions = route.load_questions()
routing = route.route_all(questions)
out_path = route.save_routing_report(routing)
print(f"Saved {out_path.name}: {len(routing)} questions")
print(routing["mode"].value_counts().to_string())
print("early abstentions:", int(routing.early_abstain.sum()))

In [ ]:
view = routing.assign(
    n_files=routing.candidate_files.map(lambda s: len([x for x in s.split(";") if x.strip()])))
view[["question_id", "wells", "dates", "mode", "n_files", "sections", "early_abstain"]]

In [ ]:
# Every early abstention and why.
routing.loc[routing.early_abstain, ["question_id", "wells", "dates", "reason"]]

## 5. Field-level absence check (`has_field`)
Structured fields only; used later to separate 'the archive has no such field' from a retrieval miss.

In [ ]:
for kind in ["cost", "personnel", "supervisor", "weather", "fluid", "activity"]:
    print(f"{kind:11s} present in any report: {store.has_field(records, kind)}")